In [1]:
%load_ext autoreload
%autoreload 2
import os
import pandas as pd
import json
os.chdir(os.path.expanduser("~/CitePretrain"))
os.getcwd()
# save_dir = "/usr/xtmp/yh386/CitePretrain/"
# change to your dir to save large files
save_dir = "~/CitePretrain/"

## 1. Download Models

In [5]:
# Download the active-index + fine-tuning checkpoint
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id="kkkevinkkk/Qwen2.5-7B-ActiveIndex-Instruct",
    local_dir=os.path.join(save_dir, "checkpoints", "Qwen2.5-7B-ActiveIndex-Instruct/"),
    repo_type="model",
)

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.87G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.33G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/823 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

'/usr/project/xtmp/yh386/CitePretrain/checkpoints/Qwen2.5-7B-ActiveIndex-Instruct'

In [3]:
# If you would like the continual-pretrained one without fine-tuning, download below one as well

snapshot_download(
    repo_id="kkkevinkkk/Qwen2.5-7B-ActiveIndex-CPT",
    local_dir=os.path.join(save_dir, "checkpoints", "Qwen2.5-7B-ActiveIndex-CPT"),
    repo_type="model",
)

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

config.json:   0%|          | 0.00/789 [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.87G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.33G [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

'/usr/project/xtmp/yh386/CitePretrain/checkpoints/Qwen2.5-7B-ActiveIndex-CPT'

## 2 Download Evaluation Data

### 2.1 Download evaluation qa files

In [6]:
from huggingface_hub import hf_hub_download
from pathlib import Path

repo_id = "kkkevinkkk/CitePretrainBench"
download_dir = Path(os.path.join(save_dir, "data", "evaluation"))

for dataset_name in ["eli5", "asqa", "sciqag", "repliqa"]:
    local_path = hf_hub_download(
        repo_id=repo_id,
        filename=f"test/{dataset_name}.jsonl",
        repo_type="dataset",
        local_dir=str(download_dir),
    )
    print(f"Downloaded to: {local_path}")

test/eli5.jsonl:   0%|          | 0.00/76.1M [00:00<?, ?B/s]

Downloaded to: /usr/xtmp/yh386/CitePretrain/data/evaluation/test/eli5.jsonl


asqa.jsonl: 0.00B [00:00, ?B/s]

Downloaded to: /usr/xtmp/yh386/CitePretrain/data/evaluation/test/asqa.jsonl


sciqag.jsonl: 0.00B [00:00, ?B/s]

Downloaded to: /usr/xtmp/yh386/CitePretrain/data/evaluation/test/sciqag.jsonl


test/repliqa.jsonl:   0%|          | 0.00/38.3M [00:00<?, ?B/s]

Downloaded to: /usr/xtmp/yh386/CitePretrain/data/evaluation/test/repliqa.jsonl


### 2.2 Download the titles space for evaluation

In [11]:
import os
from huggingface_hub import hf_hub_download

local_path = hf_hub_download(
    repo_id="kkkevinkkk/CitePretrainBench",
    filename="total_doc_ids.jsonl",
    repo_type="dataset",
    local_dir=os.path.join(save_dir, "data", "evaluation"),
)

print(local_path)

total_doc_ids.jsonl:   0%|          | 0.00/11.3M [00:00<?, ?B/s]

/usr/xtmp/yh386/CitePretrain/data/evaluation/total_doc_ids.jsonl


### 2.3 Download the knowledge source (corpus) for all datasets

In [3]:
import gzip
import shutil
from pathlib import Path
from huggingface_hub import hf_hub_download


def download_file(
    repo_id: str,
    remote_path: str,
    output_path: str | Path,
    repo_type: str = "dataset",
    unzip: bool = True,
    remove_compressed: bool = True,
) -> Path:
    """
    Download a file from Hugging Face Hub to a specific output path.

    If unzip=True and the remote file ends with .gz, it will be decompressed
    to output_path. Otherwise the downloaded file will be copied to output_path.

    Args:
        repo_id: Hugging Face repo id, e.g. "kkkevinkkk/CitePretrainBench"
        remote_path: Path in the HF repo, e.g. "ft_citation/mixed/dev_ic2_qwen.jsonl.gz"
        output_path: Final desired local path
        repo_type: Usually "dataset" or "model"
        unzip: Whether to decompress .gz files
        remove_compressed: Whether to delete the downloaded .gz after unzip/copy

    Returns:
        Path to the final output file
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    downloaded_path = Path(
        hf_hub_download(
            repo_id=repo_id,
            filename=remote_path,
            repo_type=repo_type,
        )
    )

    if unzip and remote_path.endswith(".gz"):
        with gzip.open(downloaded_path, "rb") as f_in, open(output_path, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)
    else:
        shutil.copyfile(downloaded_path, output_path)

    if remove_compressed and downloaded_path.exists():
        try:
            downloaded_path.unlink()
        except OSError:
            pass

    return output_path

In [10]:
repo_id = "kkkevinkkk/CitePretrainBench"
knowledge_sources = ["wikipedia", "sciqag", "repliqa", "common_crawl", "gpt"]

for dataset_name in knowledge_sources:
    local_path = download_file(
        repo_id=repo_id,
        remote_path=f"knowledge_source/{dataset_name}/docs.jsonl.gz",
        output_path=Path(save_dir) / "data" / "knowledge_source" / dataset_name / "docs.jsonl",
        repo_type="dataset",
        unzip=True,
    )
    print(f"Downloaded to: {local_path}")

knowledge_source/wikipedia/docs.jsonl.gz:   0%|          | 0.00/211M [00:00<?, ?B/s]

Downloaded and unzipped to: /usr/xtmp/yh386/CitePretrain/data/knowledge_source/wikipedia/docs.jsonl


knowledge_source/sciqag/docs.jsonl.gz:   0%|          | 0.00/184M [00:00<?, ?B/s]

Downloaded and unzipped to: /usr/xtmp/yh386/CitePretrain/data/knowledge_source/sciqag/docs.jsonl


knowledge_source/repliqa/docs.jsonl.gz:   0%|          | 0.00/21.5M [00:00<?, ?B/s]

Downloaded and unzipped to: /usr/xtmp/yh386/CitePretrain/data/knowledge_source/repliqa/docs.jsonl


knowledge_source/common_crawl/docs.jsonl(…):   0%|          | 0.00/299M [00:00<?, ?B/s]

Downloaded and unzipped to: /usr/xtmp/yh386/CitePretrain/data/knowledge_source/common_crawl/docs.jsonl


knowledge_source/gpt/docs.jsonl.gz:   0%|          | 0.00/5.51M [00:00<?, ?B/s]

Downloaded and unzipped to: /usr/xtmp/yh386/CitePretrain/data/knowledge_source/gpt/docs.jsonl


## 3. Download Processed Training Data

### 3.1 Download continual pretraining data

In [14]:
repo_id = "kkkevinkkk/CitePretrainBench"

mapping = {
    "pretrain/mixed/train_qwen.jsonl.gz": Path(save_dir) / "data" / "pretrain" / "mixed" / "train_qwen.jsonl",

    ## dev set is placeholder, as during continual-pretraining, we just fixed epochs and no early stopping
    "pretrain/mixed/dev_qwen.jsonl.gz": Path(save_dir) / "data" / "pretrain" / "mixed" / "dev_qwen.jsonl",
}

for remote_path, output_path in mapping.items():
    local_path = download_file(
        repo_id=repo_id,
        remote_path=remote_path,
        output_path=output_path,
        repo_type="dataset",
        unzip=True,
    )
    print(f"Downloaded to: {local_path}")

pretrain/mixed/train_qwen.jsonl.gz:   0%|          | 0.00/4.57G [00:00<?, ?B/s]

Downloaded to: /usr/xtmp/yh386/CitePretrain/data/pretrain/mixed/train_qwen.jsonl


pretrain/mixed/dev_qwen.jsonl.gz:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

Downloaded to: /usr/xtmp/yh386/CitePretrain/data/pretrain/mixed/dev_qwen.jsonl


In [ ]:
from utils import tokenize_and_save_data
model_name = "kkkevinkkk/Qwen2.5-7B-citation-raw"
## tokenize them into bin file for training as training accept .bin file
tokenize_and_save_data(os.path.join(save_dir, "data", "pretrain", "mixed", "train_qwen.jsonl"), model_name=model_name)
tokenize_and_save_data(os.path.join(save_dir, "data", "pretrain", "mixed", "dev_qwen.jsonl"), model_name=model_name)

INFO: Pandarallel will run on 32 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.
Reading /usr/xtmp/yh386/CitePretrain/data/pretrain/mixed/train_qwen.jsonl


100%|█████████████████████████████████████████████████████████████████████| 17/17 [35:49<00:00, 126.47s/it]


/usr/xtmp/yh386/CitePretrain/data/pretrain/mixed/train_qwen.bin


writing /usr/xtmp/yh386/CitePretrain/data/pretrain/mixed/train_qwen.bin: 100%|█| 1024/1024 [35:35<00:00,  2


INFO: Pandarallel will run on 32 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.
Reading /usr/xtmp/yh386/CitePretrain/data/pretrain/mixed/dev_qwen.jsonl


  0%|                                                                                | 0/1 [00:00<?, ?it/s]

### 3.2 Download fine-tuning data

In [15]:
repo_id = "kkkevinkkk/CitePretrainBench"

mapping = {
    "ft_citation/mixed/train_qwen.jsonl.gz": Path(save_dir) / "data" / "ft_citation" / "mixed" / "train_qwen.jsonl",
    "ft_citation/mixed/dev_qwen.jsonl.gz": Path(save_dir) / "data" / "ft_citation" / "mixed" / "dev_qwen.jsonl",
}

for remote_path, output_path in mapping.items():
    local_path = download_file(
        repo_id=repo_id,
        remote_path=remote_path,
        output_path=output_path,
        repo_type="dataset",
        unzip=True,
    )
    print(f"Downloaded to: {local_path}")

ft_citation/mixed/train_qwen.jsonl.gz:   0%|          | 0.00/1.41M [00:00<?, ?B/s]

Downloaded to: /usr/xtmp/yh386/CitePretrain/data/ft_citation/mixed/train_qwen.jsonl


ft_citation/mixed/dev_qwen.jsonl.gz:   0%|          | 0.00/321k [00:00<?, ?B/s]

Downloaded to: /usr/xtmp/yh386/CitePretrain/data/ft_citation/mixed/dev_qwen.jsonl


## 4. Preprocess Continual Pre-train Data by Yourself (Optional)

### 4.1 Mix docs from different knowledge sources

In [61]:
from utils import read_jsonl
import pandas as pd
knowledge_sources = ["wikipedia", "sciqag", "repliqa", "common_crawl", "gpt"]

source_dir = os.path.join(save_dir, "data/knowledge_source")
df = []
for source in knowledge_sources:
    source_path = os.path.join(source_dir, source, "docs.jsonl")
    df.append(read_jsonl(source_path))

df = pd.concat(df, ignore_index=True)
assert df["title"].duplicated().sum() == 0

print(f"Total {len(df)} documents")
df["raw_doc"] = df.apply(lambda x: {"title": x["title"], "text": x["text"], "id": x["_id"]}, axis=1)
df_titles = df[["title", "_id"]]
from utils import save_jsonl
save_pretrain_dir = os.path.join(save_dir, "data/pretrain/mixed_customized")
save_jsonl(df[["raw_doc", "source"]], os.path.join(save_pretrain_dir, "docs.jsonl"))
save_jsonl(df_titles, os.path.join(save_pretrain_dir, "total_doc_ids.jsonl"))
print(df["title"].duplicated().sum())

Reading /usr/xtmp/yh386/CitePretrain/data/knowledge_source/wikipedia/docs.jsonl
Reading /usr/xtmp/yh386/CitePretrain/data/knowledge_source/sciqag/docs.jsonl
Reading /usr/xtmp/yh386/CitePretrain/data/knowledge_source/repliqa/docs.jsonl
Reading /usr/xtmp/yh386/CitePretrain/data/knowledge_source/common_crawl/docs.jsonl
Reading /usr/xtmp/yh386/CitePretrain/data/knowledge_source/gpt/docs.jsonl
Total 99668 documents
Created directory /usr/xtmp/yh386/CitePretrain/data/pretrain/mixed_customized
Saved to /usr/xtmp/yh386/CitePretrain/data/pretrain/mixed_customized/docs.jsonl
Saved to /usr/xtmp/yh386/CitePretrain/data/pretrain/mixed_customized/total_doc_ids.jsonl
0


### 4.2 Process the mixed doc into chunks and apply training template without augmented data (Passive Index)

In [5]:
from utils import chunk_raw_doc
from transformers import AutoTokenizer
from pandarallel import pandarallel
from utils import save_jsonl, read_jsonl
import pandas as pd


pandarallel.initialize(progress_bar=False, nb_workers=64)

INFO: Pandarallel will run on 64 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [62]:
# model_name = "meta-llama/Meta-Llama-3.1-8B"
# model_suffix = "_llama3"
model_name = "kkkevinkkk/Qwen2.5-7B-citation-raw"
model_suffix = "_qwen"
tokenizer = AutoTokenizer.from_pretrained(model_name)

df_doc = df[["raw_doc", "source"]]

chunk_length = 768
df_doc["raw_chunks"] = df_doc.parallel_apply(lambda x: chunk_raw_doc(x, tokenizer=tokenizer, max_len=chunk_length, modify_title=False), axis=1)

df_chunks = pd.DataFrame({"raw_doc": df_doc["raw_chunks"].explode()}).reset_index(drop=True)


INFO: Pandarallel will run on 64 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


Token indices sequence length is longer than the specified maximum sequence length for this model (182243 > 131072). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (137759 > 131072). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (181210 > 131072). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (185671 > 131072). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (155867 > 131072). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified m

In [70]:
from utils import generate_pretrain_doc, sample_multi_granular_chunks, read_jsonl
import numpy as np
import pandas as pd


reverse_order = True
add_pretrain_special_tokens = True
title_special_tokens = "reserved"
no_title = False
multi_granular = False

df_pretrain = df_chunks
if multi_granular:
    np.random.seed(42)
    df_pretrain["raw_doc"] = df_pretrain.parallel_apply(lambda x: sample_multi_granular_chunks(x), axis=1)
    df_pretrain = df_pretrain.explode("raw_doc").reset_index(drop=True)

df_pretrain = pd.DataFrame(df_pretrain)
df_pretrain["text"] = df_pretrain.parallel_apply(lambda x: generate_pretrain_doc(x,tokenizer, reverse_order=reverse_order, add_pretrain_special_tokens=add_pretrain_special_tokens, title_special_tokens=title_special_tokens, no_title=no_title), axis=1)
save_path = os.path.join(save_dir, f"data/pretrain/mixed_customized/train_passive-index{model_suffix}.jsonl")
save_jsonl(df_pretrain[['text']], save_path)

Saved to /usr/xtmp/yh386/CitePretrain/data/pretrain/mixed_customized/train_passive-index_qwen.jsonl


In [71]:
from utils import tokenize_and_save_data
## tokenize them into bin file for training
tokenize_and_save_data(save_path, model_name=model_name)

INFO: Pandarallel will run on 32 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.
Reading /usr/xtmp/yh386/CitePretrain/data/pretrain/mixed_customized/train_passive-index_qwen.jsonl


100%|████████████████████████████████████████████████████████████████████████| 2/2 [01:50<00:00, 55.15s/it]


/usr/xtmp/yh386/CitePretrain/data/pretrain/mixed_customized/train_passive-index_qwen.bin


writing /usr/xtmp/yh386/CitePretrain/data/pretrain/mixed_customized/train_passive-index_qwen.bin: 100%|█| 1


### 4.3 Download Active Index Data

In [4]:
repo_id = "kkkevinkkk/CitePretrainBench"

mapping = {
    # "pretrain/backward/train.jsonl.gz": Path(save_dir) / "data" / "pretrain" / "backward" / "train.jsonl",
    "pretrain/forward/train.jsonl.gz": Path(save_dir) / "data" / "pretrain" / "forward" / "train.jsonl",
}

for remote_path, output_path in mapping.items():
    local_path = download_file(
        repo_id=repo_id,
        remote_path=remote_path,
        output_path=output_path,
        repo_type="dataset",
        unzip=True,
    )
    print(f"Downloaded to: {local_path}")

pretrain/forward/train.jsonl.gz:   0%|          | 0.00/1.67G [00:00<?, ?B/s]

Downloaded to: /usr/xtmp/yh386/CitePretrain/data/pretrain/forward/train.jsonl


In [6]:
# model_name = "meta-llama/Meta-Llama-3.1-8B"
# model_suffix = "_llama3"
model_name = "kkkevinkkk/Qwen2.5-7B-citation-raw"
model_suffix = "_qwen"
tokenizer = AutoTokenizer.from_pretrained(model_name)

df_pretrain = read_jsonl(os.path.join(save_dir, f"data/pretrain/mixed_customized/train_passive-index{model_suffix}.jsonl"))



Reading /usr/xtmp/yh386/CitePretrain/data/pretrain/mixed_customized/train_passive-index_qwen.jsonl


In [7]:
df_backward = read_jsonl(os.path.join(save_dir, f"data/pretrain/backward/train.jsonl"))
df_backward['text'] = df_backward['text'].apply(lambda x: x + tokenizer.eos_token)
df_forward = read_jsonl(os.path.join(save_dir, f"data/pretrain/forward/train.jsonl"))
df_forward['text'] = df_forward['text'].apply(lambda x: x + tokenizer.eos_token)

Reading /usr/xtmp/yh386/CitePretrain/data/pretrain/backward/train.jsonl
Reading /usr/xtmp/yh386/CitePretrain/data/pretrain/forward/train.jsonl


In [8]:
print(df_forward['text'][0])
print("")
print(df_backward['text'][0])

**Question 1:**  
How is Brian Conaghan related to the Freedom theme of the Edinburgh International Book Festival 2018: Freedom Programme for Young People?

**Answer 1:**  
Brian Conaghan is directly related to the Freedom theme of the Edinburgh International Book Festival 2018: Freedom Programme for Young People through his involvement in a debate at the festival. This debate, titled "The Importance of Writing Sensitive Male Characters," is part of the programme aimed at exploring the concept of freedom, particularly in the context of children and young adults’ experiences and challenges. Conaghan’s participation demonstrates the festival’s commitment to discussing and addressing themes of freedom and identity within the young audience, especially in relation to sensitive topics such as heroin addiction and masculinity.

**Question 2:**  
What role does Brian Conaghan play in the Children’s & Education Programme at the Edinburgh International Book Festival 2018, as outlined in the doc

In [9]:
df_total = pd.concat([df_pretrain[['text']], df_forward[['text']], df_backward[['text']]])

In [10]:
save_path = os.path.join(save_dir, f"data/pretrain/mixed_customized/train_active-index{model_suffix}.jsonl")
save_jsonl(df_total[['text']], save_path)

Saved to /usr/xtmp/yh386/CitePretrain/data/pretrain/mixed_customized/train_active-index_qwen.jsonl


In [ ]:
from utils import tokenize_and_save_data
## tokenize them into bin file for training
tokenize_and_save_data(save_path, model_name=model_name)

INFO: Pandarallel will run on 32 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.
Reading /usr/xtmp/yh386/CitePretrain/data/pretrain/mixed_customized/train_active-index_qwen.jsonl


 29%|████████████████████▉                                                  | 5/17 [08:04<18:49, 94.09s/it]

## 5. Preprocess Fine-tune Data by Yourself (Optional)

### 5.1 Download Fine-tune QA by Dataset

In [41]:
repo_id = "kkkevinkkk/CitePretrainBench"
download_root = Path(save_dir) / "data" / "ft_citation"

for dataset_name in ["eli5", "asqa", "sciqag", "repliqa", "hotpotqa_medium"]:
    for split in ["train", "dev"]:
        local_path = download_file(
            repo_id=repo_id,
            remote_path=f"ft_citation/{dataset_name}/{split}.jsonl.gz",
            output_path=download_root / dataset_name / f"{split}.jsonl",
            repo_type="dataset",
            unzip=True,
        )
        print(f"Downloaded to: {local_path}")

ft_citation/eli5/train.jsonl.gz:   0%|          | 0.00/5.71M [00:00<?, ?B/s]

Downloaded to: /usr/xtmp/yh386/CitePretrain/data/ft_citation/eli5/train.jsonl


ft_citation/eli5/dev.jsonl.gz:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

Downloaded to: /usr/xtmp/yh386/CitePretrain/data/ft_citation/eli5/dev.jsonl


ft_citation/asqa/train.jsonl.gz:   0%|          | 0.00/2.70M [00:00<?, ?B/s]

...

Downloaded to: /usr/xtmp/yh386/CitePretrain/data/ft_citation/repliqa/dev.jsonl


ft_citation/hotpotqa_medium/train.jsonl.(…):   0%|          | 0.00/4.27M [00:00<?, ?B/s]

Downloaded to: /usr/xtmp/yh386/CitePretrain/data/ft_citation/hotpotqa_medium/train.jsonl


ft_citation/hotpotqa_medium/dev.jsonl.gz:   0%|          | 0.00/1.41M [00:00<?, ?B/s]

Downloaded to: /usr/xtmp/yh386/CitePretrain/data/ft_citation/hotpotqa_medium/dev.jsonl


### 5.2 Process each dataset into fine-tune format

In [42]:
from utils import read_jsonl, save_jsonl
from prompter import Prompter

from pandarallel import pandarallel
pandarallel.initialize(progress_bar=False, nb_workers=64)



INFO: Pandarallel will run on 64 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.


In [43]:
from transformers import AutoTokenizer
# model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
model_name = "kkkevinkkk/Qwen2.5-7B-citation"
tokenizer = AutoTokenizer.from_pretrained(model_name)


if "kkkevinkkk/Qwen2.5-7B-citation" in model_name:
    model_suffix = "_qwen"
else:
    model_suffix = "_llama3"


In [51]:

for dataset_name in ["eli5", "asqa", "sciqag", "repliqa", "hotpotqa_medium"]:
    df_train = read_jsonl(os.path.join(save_dir, f"data/ft_citation/{dataset_name}/train.jsonl"))
    df_dev = read_jsonl(os.path.join(save_dir, f"data/ft_citation/{dataset_name}/dev.jsonl"))

    task_type = "citation_ic2"
    prompter = Prompter(model_name=model_name, n_doc=0, n_shot=0, dataset_name=dataset_name)


    def get_citation_data(row):
        eos_token = prompter.tokenizer.eos_token

        text_input = prompter.generate_text_input(eval_item=row, task_type=task_type, faithful_type=None)
        output = row["output"] + eos_token
        row["input"] = text_input
        row["output"] = output
        return row


    df_train = df_train.parallel_apply(lambda x: get_citation_data(x), axis=1)
    df_dev = df_dev.parallel_apply(lambda x: get_citation_data(x), axis=1)
    print("Input:", df_train["input"][0])
    print("")
    print("Output:", df_train["output"][0])
    save_jsonl(df_train, os.path.join(save_dir, f"data/ft_citation/{dataset_name}", f"train{model_suffix}.jsonl"))
    save_jsonl(df_dev, os.path.join(save_dir, f"data/ft_citation/{dataset_name}", f"dev{model_suffix}.jsonl"))

Reading /usr/xtmp/yh386/CitePretrain/data/ft_citation/eli5/train.jsonl
Reading /usr/xtmp/yh386/CitePretrain/data/ft_citation/eli5/dev.jsonl
Input: <|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Instruction: Write an accurate, engaging, and concise answer for the given question. Use an unbiased and journalistic tone. For every factual claim, cite the source immediately after the claim using the following format: <|reserved_special_token_0|> title <|reserved_special_token_1|>, where title is the title of the document that supports the claim.

Question: What is an electric boogaloo?

Answer:<|im_end|>
<|im_start|>assistant





Output: "Electric Boogaloo" originally refers to the 1984 movie *Breakin' 2: Electric Boogaloo*, which is the sequel to the breakdancing film *Breakin'*.<|reserved_special_token_0|>Understanding Electric Boogaloo: Dance and Pop Culture Phenomenon by Dictionary.com<|reserved_special_token_1|> The sequel *Breakin' 2: Electric Boogaloo* was

### 5.3  Mix different datasets into one fine-tuning data

In [52]:
from utils import read_jsonl, save_jsonl
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=False)
train_data_num_per_dataset = 1000
dev_data_num_per_dataset = 200

# data_path_template = "/usr/xtmp/yh386/citation/datasets/{dataset_name}/ft_citation/{split}{model_suffix}.jsonl"
data_path_template = os.path.join(save_dir, "data/ft_citation/{dataset_name}", "{split}{model_suffix}.jsonl")

data_path_template = data_path_template.replace("{model_suffix}", model_suffix)
df_train = []
df_dev = []
for dataset_name in ["hotpotqa_medium", "eli5", "asqa", "repliqa", "sciqag"]:
# for dataset_name in ["eli5"]:
    df_train_ = read_jsonl(data_path_template.format(dataset_name=dataset_name, split="train"))
    df_train_["total_len"] = df_train_.parallel_apply(lambda x: len(tokenizer(x["input"])["input_ids"]) + len(tokenizer(x["output"])["input_ids"]), axis=1)
    df_train_ = df_train_[df_train_["total_len"] < 400].sample(n=train_data_num_per_dataset, random_state=42)[["input", "output"]]
    df_dev_ = read_jsonl(data_path_template.format(dataset_name=dataset_name, split="dev")).sample(n=dev_data_num_per_dataset, random_state=42)[["input", "output"]]
    df_train.append(df_train_)
    df_dev.append(df_dev_)
df_train = pd.concat(df_train)
df_dev = pd.concat(df_dev)



INFO: Pandarallel will run on 64 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.
Reading /usr/xtmp/yh386/CitePretrain/data/ft_citation/hotpotqa_medium/train_qwen.jsonl
Reading /usr/xtmp/yh386/CitePretrain/data/ft_citation/hotpotqa_medium/dev_qwen.jsonl
Reading /usr/xtmp/yh386/CitePretrain/data/ft_citation/eli5/train_qwen.jsonl
Reading /usr/xtmp/yh386/CitePretrain/data/ft_citation/eli5/dev_qwen.jsonl
Reading /usr/xtmp/yh386/CitePretrain/data/ft_citation/asqa/train_qwen.jsonl
Reading /usr/xtmp/yh386/CitePretrain/data/ft_citation/asqa/dev_qwen.jsonl
Reading /usr/xtmp/yh386/CitePretrain/data/ft_citation/repliqa/train_qwen.jsonl
Reading /usr/xtmp/yh386/CitePretrain/data/ft_citation/repliqa/dev_qwen.jsonl
Reading /usr/xtmp/yh386/CitePretrain/data/ft_citation/sciqag/train_qwen.jsonl
Reading /usr/xtmp/yh386/CitePretrain/data/ft_citation/sciqag/dev_qwen.jsonl


In [58]:
df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)
df_dev = df_dev.sample(frac=1, random_state=42).reset_index(drop=True)
print(df_train["input"][1])
print(df_train["output"][1])

save_path = os.path.join(save_dir, "data", "ft_citation", "mixed_customized")
save_jsonl(df_train, os.path.join(save_path, f"train{model_suffix}.jsonl"))
save_jsonl(df_dev, os.path.join(save_path, f"dev{model_suffix}.jsonl"))

Created directory /usr/xtmp/yh386/CitePretrain/data/ft_citation/mixed_customized
Saved to /usr/xtmp/yh386/CitePretrain/data/ft_citation/mixed_customized/train_qwen.jsonl
Saved to /usr/xtmp/yh386/CitePretrain/data/ft_citation/mixed_customized/dev_qwen.jsonl
